<a href="https://colab.research.google.com/github/fatmasenguler/Mutation_KRAS_analysis/blob/main/Table_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython networkx

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 6GOD.pdb to 6GOD.pdb
Saving 6GOF.pdb to 6GOF.pdb


In [ ]:
"""
channel_convergence.py

Convergence check for every channel BEFORE trusting any occupancy / heat-capacity
number computed from its truncated path ensemble. Reproduces the Figure 2
methodology of the manuscript exactly.

Ground truth (exact, all spanning trees, all path lengths, one matrix op):
    R_ab(full) = K†_aa + K†_bb - 2 K†_ab
where K† is the Moore-Penrose pseudoinverse of the FULL weighted Laplacian
(edge conductance w_ij = exp(-d_ij / kT)).

Truncated subset at max path length L:
    enumerate all simple paths of node length <= L between a and b;
    take the UNION of visited nodes;
    build the INDUCED subgraph G(L): every edge of G between two visited nodes
    (not just the edges lying on a path -- shortcut edges carry parallel current);
    R_ab(L) = Kirchhoff pseudoinverse of G(L).

Convergence ratio (manuscript / Figure 2 convention):
    r(L) = R_ab(L) / R_ab(full)   >= 1,  decreasing monotonically to 1 (Rayleigh)
    recovery(L) = 100 / r(L)  [%]  <- this is the "93-99%" language in the caption

Acceptance at L = 9:
    r(9) < 1.01           -> PASS  (within 1%, strict threshold)
    93% <= recovery(9)    -> MARGINAL (report the exact %, as the paper does)
    recovery(9) < 93%     -> FAIL   (extend L, or drop the channel from the table)

A channel with very few paths (e.g. 12->170) is the prime suspect for FAIL:
its induced subgraph stays sparse and R_ab(L) does not descend to R_ab(full).

Run:  edit WT_PDB / MUT_PDB / CHANNELS / params, then `python channel_convergence.py`
"""

import numpy as np
import networkx as nx
from Bio.PDB import PDBParser
from numpy.linalg import pinv

# ----------------------- USER SETTINGS -----------------------
WT_PDB   = "6GOD.pdb"
MUT_PDB  = "6GOF.pdb"
CUTOFF   = 7.8
kT       = 1.0
LMIN, LMAX = 3, 10          # report r(L) for L in this inclusive range

# check EVERYTHING at one cutoff: original 5 + new 5
CHANNELS = [
    # original manuscript channels
    (6, 11), (55, 60), (110, 117), (141, 146), (19, 142),
    # new corridors
    (12, 35), (12, 61), (12, 156), (12, 170), (35, 61),
]
MAKE_PLOT = True            # save channel_convergence.png (Figure-2 style)
# -------------------------------------------------------------


def build_ca_graph(pdb_file, cutoff):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("prot", pdb_file)
    chain = list(structure[0].get_chains())[0]
    coords, res_ids = [], []
    for residue in chain:
        if "CA" in residue:
            coords.append(residue["CA"].coord)
            res_ids.append(residue.get_id()[1])
    coords = np.array(coords)
    n = len(coords)
    G = nx.Graph()
    G.add_nodes_from(res_ids)
    for i in range(n):
        for j in range(i + 1, n):
            d = np.linalg.norm(coords[i] - coords[j])
            if d <= cutoff:
                G.add_edge(res_ids[i], res_ids[j], weight=d)
    return G


def weighted_pinv(G, kT):
    """Moore-Penrose pseudoinverse of the weighted Laplacian (conductance exp(-d/kT))."""
    ids = list(G.nodes())
    idx = {r: i for i, r in enumerate(ids)}
    n = len(ids)
    L = np.zeros((n, n))
    for u, v, data in G.edges(data=True):
        w = np.exp(-data['weight'] / kT)
        i, j = idx[u], idx[v]
        L[i, i] += w; L[j, j] += w; L[i, j] -= w; L[j, i] -= w
    return pinv(L), idx


def R_from(K, idx, a, b):
    ia, ib = idx[a], idx[b]
    return K[ia, ia] + K[ib, ib] - 2.0 * K[ia, ib]


def find_paths_of_length(G, start, end, length):
    paths, stack = [], [(start, [start])]
    while stack:
        node, path = stack.pop()
        if len(path) == length:
            if node == end:
                paths.append(path)
            continue
        if len(path) < length:
            for nb in sorted(G.neighbors(node)):
                if nb not in path:
                    stack.append((nb, path + [nb]))
    return paths


def convergence(G, a, b, kT, lmin, lmax):
    """Return R_full and a list of (L, R_L, r, recovery%) rows."""
    Kf, idxf = weighted_pinv(G, kT)
    R_full = R_from(Kf, idxf, a, b)
    visited = set()
    rows = []
    for Ln in range(lmin, lmax + 1):
        for p in find_paths_of_length(G, a, b, Ln):
            visited.update(p)
        if a in visited and b in visited and len(visited) >= 2:
            H = G.subgraph(visited)
            Ks, idxs = weighted_pinv(H, kT)
            R_L = R_from(Ks, idxs, a, b)
            r = R_L / R_full if R_full != 0 else float('nan')
            rec = 100.0 / r if (np.isfinite(r) and r != 0) else float('nan')
        else:
            R_L, r, rec = float('nan'), float('nan'), float('nan')
        rows.append((Ln, R_L, r, rec))
    return R_full, rows


def verdict(rows, L_target=9):
    """PASS / MARGINAL / FAIL based on r and recovery at L_target."""
    row = next((x for x in rows if x[0] == L_target), None)
    if row is None or not np.isfinite(row[2]):
        return "N/A", float('nan'), float('nan')
    r, rec = row[2], row[3]
    if r < 1.01:
        return "PASS", r, rec
    if rec >= 93.0:
        return "MARGINAL", r, rec
    return "FAIL", r, rec


def main():
    print(f"Building graphs (cutoff={CUTOFF}) ...")
    G_wt = build_ca_graph(WT_PDB, CUTOFF)
    G_mut = build_ca_graph(MUT_PDB, CUTOFF)

    out = [f"Channel convergence  r(L) = R_ab(L)/R_ab(full)   (>=1, ->1)",
           f"cutoff={CUTOFF}, kT={kT}, L={LMIN}-{LMAX}. recovery% = 100/r.",
           f"Accept at L=9:  r<1.01 PASS | recovery>=93% MARGINAL | else FAIL",
           ""]

    results = {}   # (a,b) -> dict(wt=..., mut=...)
    for (a, b) in CHANNELS:
        print(f"\nChannel {a} -> {b} ...")
        R_full_wt, rows_wt = convergence(G_wt, a, b, kT, LMIN, LMAX)
        R_full_mut, rows_mut = convergence(G_mut, a, b, kT, LMIN, LMAX)
        results[(a, b)] = dict(wt=rows_wt, mut=rows_mut,
                               Rwt=R_full_wt, Rmut=R_full_mut)

        vw, rw, rcw = verdict(rows_wt)
        vm, rm, rcm = verdict(rows_mut)
        out.append(f"Res {a} -> {b}   R_full: WT={R_full_wt:.3f}  G12D={R_full_mut:.3f}")
        out.append(f"  {'L':>3}{'r_WT':>9}{'rec%_WT':>10}{'r_G12D':>10}{'rec%_G12D':>11}")
        for (Lw, _, rW, recW), (_, _, rM, recM) in zip(rows_wt, rows_mut):
            def f(x): return f"{x:.3f}" if np.isfinite(x) else "  --"
            def g(x): return f"{x:.1f}" if np.isfinite(x) else "  --"
            out.append(f"  {Lw:>3}{f(rW):>9}{g(recW):>10}{f(rM):>10}{g(recM):>11}")
        out.append(f"  verdict@L=9:  WT={vw} (r={rw:.3f}, rec={rcw:.1f}%)   "
                   f"G12D={vm} (r={rm:.3f}, rec={rcm:.1f}%)")
        out.append("")

    text = "\n".join(out)
    print("\n" + text)
    with open("channel_convergence.txt", "w", encoding="utf-8") as f:
        f.write(text)
    print("Saved: channel_convergence.txt")

    if MAKE_PLOT:
        try:
            import matplotlib.pyplot as plt
            nch = len(CHANNELS)
            ncol = min(5, nch)
            nrow = int(np.ceil(nch / ncol))
            fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 2.8 * nrow),
                                     squeeze=False)
            for ax_i, (a, b) in enumerate(CHANNELS):
                ax = axes[ax_i // ncol][ax_i % ncol]
                rw = results[(a, b)]['wt']
                rm = results[(a, b)]['mut']
                Lw = [x[0] for x in rw]
                ax.plot(Lw, [x[2] for x in rw], 'o-', color='tab:blue',
                        label='WT (6GOD)', ms=5)
                ax.plot(Lw, [x[2] for x in rm], 's--', color='tab:red',
                        label='G12D (6GOF)', ms=5)
                ax.axhline(1.0, color='k', lw=0.8)
                ax.axhline(1.01, color='tab:green', ls=':', lw=1.2,
                           label='1% threshold')
                ax.set_title(f"{a}\u2192{b}", fontsize=10)
                ax.set_xlabel("Max path length L")
                if ax_i % ncol == 0:
                    ax.set_ylabel("r(L) = R(L)/R(full)")
            axes[0][ncol - 1].legend(fontsize=7)
            for k in range(nch, nrow * ncol):
                axes[k // ncol][k % ncol].axis('off')
            fig.suptitle(f"Channel convergence  (cutoff {CUTOFF} \u00c5, kT={kT})",
                         fontsize=12)
            fig.tight_layout(rect=[0, 0, 1, 0.96])
            fig.savefig("channel_convergence.png", dpi=200)
            print("Saved: channel_convergence.png")
        except Exception as e:
            print(f"(plot skipped: {e})")


if __name__ == "__main__":
    main()